# Basic RAG

## Simple idea

RAG = **Retrieval-Augmented Generation**.

Instead of:
`Question → LLM → Answer`

we do:
`Documents → Chunks → Embeddings → Vector DB → Retrieve → Context → LLM → Answer`

The supplied notebooks follow this same path.

In [ ]:
%pip install -q sentence-transformers chromadb google-genai

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("GEMINI_API_KEY: ")

from google import genai
from sentence_transformers import SentenceTransformer
import chromadb

llm = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
embedder = SentenceTransformer("all-MiniLM-L6-v2")

## 1. Documents → chunks

A chunk is a small piece of a document. Overlap helps preserve meaning across boundaries.

In [ ]:
documents = [
    {"source":"python.txt",
     "text":"Python is a high-level programming language. It is used for web development, data science, machine learning, automation and AI."},
    {"source":"rag.txt",
     "text":"RAG means Retrieval-Augmented Generation. It retrieves useful information from an external knowledge base and gives it to a language model as context."},
    {"source":"ml.txt",
     "text":"Machine learning is a branch of artificial intelligence. Common types are supervised, unsupervised and reinforcement learning."},
]

def chunk_text(text, size=120, overlap=20):
    out, start = [], 0
    while start < len(text):
        piece = text[start:start+size].strip()
        if piece:
            out.append(piece)
        start += size - overlap
    return out

chunks = []
for d in documents:
    for i, piece in enumerate(chunk_text(d["text"])):
        chunks.append({"source":d["source"], "chunk_id":i, "text":piece})

print("chunks:", len(chunks))

## 2. Chunks → embeddings

An embedding is a vector of numbers representing text meaning.

In [ ]:
texts = [x["text"] for x in chunks]
vectors = embedder.encode(texts, normalize_embeddings=True)

print("vector count:", len(vectors))
print("vector size:", vectors.shape[1])

## 3. Embeddings → vector database

Chroma stores vectors plus the original text and metadata.

In [ ]:
client = chromadb.Client()
collection = client.get_or_create_collection("basic_rag")

collection.add(
    ids=[f"{x['source']}-{x['chunk_id']}" for x in chunks],
    documents=texts,
    embeddings=vectors.tolist(),
    metadatas=[{"source":x["source"], "chunk_id":x["chunk_id"]} for x in chunks],
)

print("stored:", collection.count())

## 4. Retrieve

The question is embedded, then the nearest chunks are returned.

In [ ]:
question = "What is RAG?"
q = embedder.encode([question], normalize_embeddings=True)

results = collection.query(
    query_embeddings=q.tolist(),
    n_results=3,
    include=["documents","metadatas","distances"],
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(meta, "distance=", round(distance,4))
    print(doc)
    print("-"*50)

## 5. Context → LLM

Retriever finds evidence. The LLM turns that evidence into a readable answer.

In [ ]:
context = "\n\n".join(
    f"[Source: {m['source']}, Chunk: {m['chunk_id']}]\n{d}"
    for d,m in zip(results["documents"][0], results["metadatas"][0])
)

prompt = (
    "Answer using only the context. If the answer is not there, say "
    "'I don't know based on the provided documents.'\n\n"
    "Context:\n" + context + "\n\nQuestion:\n" + question
)

answer = llm.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
)

print(answer.text)

## 6. Simple evaluation

First check retrieval.

`Expected source appears in top-k = PASS`

This is a useful sanity check, but not a complete RAG evaluation.

In [ ]:
tests = [
    ("What does RAG stand for?", "rag.txt"),
    ("What is Python used for?", "python.txt"),
    ("What are common types of machine learning?", "ml.txt"),
]

for question, expected in tests:
    q = embedder.encode([question], normalize_embeddings=True)
    r = collection.query(query_embeddings=q.tolist(), n_results=3)
    sources = [m["source"] for m in r["metadatas"][0]]
    print("PASS" if expected in sources else "FAIL", question, sources)

## Improvement checklist

Change one thing at a time:
- chunk size
- chunk overlap
- embedding model
- top-k
- metadata filters
- prompt

Then measure again.

Mental model:

`Documents → Chunks → Embeddings → Vector DB → Retrieval → Context → LLM`